In [7]:
import pandas as pd
import numpy as np
import mysql.connector

# ---- connect to MySQL ----
conn = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password="Anvitha@0109",  # put your MySQL root password here
    database="retail_forecasting",
)

cur = conn.cursor()

# ---- create output tables if they do not already exist ----
cur.execute("""
CREATE TABLE IF NOT EXISTS forecast_results (
    id INT AUTO_INCREMENT PRIMARY KEY,
    dataset VARCHAR(20) NOT NULL,
    entity_id VARCHAR(100) NOT NULL,
    week_start VARCHAR(30) NOT NULL,
    model_name VARCHAR(20) NOT NULL,
    y_true DOUBLE NOT NULL,
    y_pred DOUBLE NOT NULL
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS metrics (
    id INT AUTO_INCREMENT PRIMARY KEY,
    dataset VARCHAR(20) NOT NULL,
    model_name VARCHAR(20) NOT NULL,
    metric_name VARCHAR(20) NOT NULL,
    metric_value DOUBLE NOT NULL,
    entity_id VARCHAR(100)
);
""")

conn.commit()

# ---- read demand data from MySQL ----
df = pd.read_sql(
    "SELECT dataset, entity_id, week_start, y FROM clean_weekly_demand",
    con=conn
)

print("Rows loaded from clean_weekly_demand:", len(df))
print(df.head())

# ---- sort time properly ----
# Walmart week_start is a real date, like 2010-02-01.
# Instacart week_start is synthetic, like week_001.

df["week_sort"] = 0

walmart_mask = df["dataset"] == "walmart"
instacart_mask = df["dataset"] == "instacart"

# Walmart: convert date to numeric YYYYMMDD
df.loc[walmart_mask, "week_sort"] = (
    pd.to_datetime(df.loc[walmart_mask, "week_start"], errors="coerce")
    .dt.strftime("%Y%m%d")
    .astype(int)
)

# Instacart: convert week_001 -> 1, week_002 -> 2, etc.
df.loc[instacart_mask, "week_sort"] = (
    df.loc[instacart_mask, "week_start"]
    .astype(str)
    .str.replace("week_", "", regex=False)
    .astype(int)
)

# Sort by dataset, entity, and time
df = df.sort_values(["dataset", "entity_id", "week_sort"]).reset_index(drop=True)

# ---- make baseline predictions ----
def add_baselines(g):
    g = g.copy()

    # Lag-1 model: previous week's demand
    g["y_pred_lag1"] = g["y"].shift(1)

    # MA4 model: average demand from previous 4 weeks
    g["y_pred_ma4"] = g["y"].rolling(4).mean().shift(1)

    return g


pred = df.groupby(["dataset", "entity_id"], group_keys=False).apply(add_baselines)

# Keep only rows where prediction exists
pred_lag1 = pred.dropna(subset=["y_pred_lag1"]).copy()
pred_ma4 = pred.dropna(subset=["y_pred_ma4"]).copy()

# ---- compute metrics ----
def compute_metrics(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)

    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    # Avoid division by zero for MAPE
    nonzero_mask = y_true != 0

    if nonzero_mask.sum() > 0:
        mape = np.mean(
            np.abs(
                (y_true[nonzero_mask] - y_pred[nonzero_mask])
                / y_true[nonzero_mask]
            )
        ) * 100
    else:
        mape = 0

    return mae, rmse, mape


metrics_rows = []

for model_name, subdf, ypred_col in [
    ("lag1", pred_lag1, "y_pred_lag1"),
    ("ma4", pred_ma4, "y_pred_ma4"),
]:
    for ds in subdf["dataset"].unique():
        tmp = subdf[subdf["dataset"] == ds]

        mae, rmse, mape = compute_metrics(tmp["y"], tmp[ypred_col])

        metrics_rows.extend([
            (ds, model_name, "MAE", float(mae), None),
            (ds, model_name, "RMSE", float(rmse), None),
            (ds, model_name, "MAPE", float(mape), None),
        ])

# ---- clear old baseline results ----
cur.execute("DELETE FROM forecast_results WHERE model_name IN ('lag1', 'ma4')")
cur.execute("DELETE FROM metrics WHERE model_name IN ('lag1', 'ma4')")
conn.commit()

# ---- insert forecast results ----
insert_forecast = """
INSERT INTO forecast_results 
(dataset, entity_id, week_start, model_name, y_true, y_pred)
VALUES (%s, %s, %s, %s, %s, %s)
"""

# Lag-1 forecast rows
lag_rows = list(
    pred_lag1[["dataset", "entity_id", "week_start", "y", "y_pred_lag1"]]
    .itertuples(index=False, name=None)
)

cur.executemany(
    insert_forecast,
    [
        (d, e, str(w), "lag1", float(y), float(p))
        for (d, e, w, y, p) in lag_rows
    ]
)

# MA4 forecast rows
ma4_rows = list(
    pred_ma4[["dataset", "entity_id", "week_start", "y", "y_pred_ma4"]]
    .itertuples(index=False, name=None)
)

cur.executemany(
    insert_forecast,
    [
        (d, e, str(w), "ma4", float(y), float(p))
        for (d, e, w, y, p) in ma4_rows
    ]
)

conn.commit()

# ---- insert metrics ----
insert_metric = """
INSERT INTO metrics 
(dataset, model_name, metric_name, metric_value, entity_id)
VALUES (%s, %s, %s, %s, %s)
"""

cur.executemany(insert_metric, metrics_rows)
conn.commit()

print("Saved forecasts:", len(lag_rows) + len(ma4_rows))
print("Saved metric rows:", len(metrics_rows))

# ---- preview results ----
print("\nForecast preview:")
print(pred_lag1[["dataset", "entity_id", "week_start", "y", "y_pred_lag1"]].head())

print("\nMetrics:")
for row in metrics_rows:
    print(row)

# ---- close connection ----
cur.close()
conn.close()

C:\Users\allaa\AppData\Local\Temp\ipykernel_54984\3875578560.py:43: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


Rows loaded from clean_weekly_demand: 423579
     dataset entity_id week_start        y
0  instacart         1   week_001  27747.0
1  instacart         1   week_002  14964.0
2  instacart         1   week_003   9207.0
3  instacart         1   week_004   5900.0
4  instacart         1   week_005   4152.0


C:\Users\allaa\AppData\Local\Temp\ipykernel_54984\3875578560.py:91: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pred = df.groupby(["dataset", "entity_id"], group_keys=False).apply(add_baselines)


Saved forecasts: 830023
Saved metric rows: 12

Forecast preview:
     dataset entity_id week_start        y  y_pred_lag1
1  instacart         1   week_002  14964.0      27747.0
2  instacart         1   week_003   9207.0      14964.0
3  instacart         1   week_004   5900.0       9207.0
4  instacart         1   week_005   4152.0       5900.0
5  instacart         1   week_006   2994.0       4152.0

Metrics:
('instacart', 'lag1', 'MAE', 6623.7744, None)
('instacart', 'lag1', 'RMSE', 27502.715940483646, None)
('instacart', 'lag1', 'MAPE', 98.78024782346614, None)
('walmart', 'lag1', 'MAE', 2102.4732758064165, None)
('walmart', 'lag1', 'RMSE', 7206.830019237782, None)
('walmart', 'lag1', 'MAPE', 149.2416010207188, None)
('instacart', 'ma4', 'MAE', 8242.20807875085, None)
('instacart', 'ma4', 'RMSE', 27834.344413751027, None)
('instacart', 'ma4', 'MAPE', 273.3912918519359, None)
('walmart', 'ma4', 'MAE', 2155.0248133979367, None)
('walmart', 'ma4', 'RMSE', 6518.654024774881, None)
('walmar

In [5]:
!pip install mysql-connector-python

^C
